# NAVSEA_ML Datasets
The purpose of this project is to allow simple, easy access to reference datasets.  This is modeled after the implementations in tensorflow.keras.datasets.  The intent is that the end user does not know or care about the underlying source or location of the data.  If the named dataset is available (cached) on the system it will be used.  If it is not available locally, it will be retrieved and unpacked from the source.

There are three steps to this process:

* The creation of a dataset
    the author will define a layout of the data that is delivered to the server for distribution.  This will probably follow the format of (x_train, y_train), (x_test, y_test)

* The hosting of a dataset
    this is the well known public endpoint for the dataset that will (probably) be hardcoded into a subclass.

* The retrieval of a dataset
    For this to work in a secure environment, users will have to subscribe to receive an API key that will be sent to the server when requesting the archive.



# Prepare a dataset
This is a one time deal and the output npz file will go in the known location


In [16]:
from glob import glob
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import os
import requests
import appdirs
from tqdm.notebook import tqdm
import os
import appdirs


#############
#
# Do whatever is necessary to build your single npz file
#
#############

species = ['norcar', 'bkcchi', 'amerob', 'blujay', 'vesspa']
manifest = """"""
#source of the data to build dataset
#root = "/C45_ML/birdsong-recognition/train_audio"
root = "../../Downloads/birdsong-recognition/train_audio"

# where are we putting it to make it public
#webroot = "/asecc_ml/gary.huntress/"
webroot = "/tmp/"

# what will we name it locally (in the cache directory)
filename = 'birdsongs.npz'

# full remote url of file to retrieve 
url = 'http://npa0mlearn01:8897/birdsongs.npz'
url = 'http://localhost:5001/birdsongs.npz'

x_all = []
y_all = []
for idx, s in enumerate(species):
    fnames = glob(f"{root}/{s}/*.mp3")
    for f in tqdm(fnames[:20]):
        x,sr = librosa.load(f)
        x_all.append(x)
        y_all.append(idx)
        
x_train, x_test, y_train, y_test = train_test_split(x_all, y_all)
x_train = np.array(x_train, dtype=object)
y_train = np.array(y_train, dtype=object)
x_test  = np.array(x_test,  dtype=object)
y_test  = np.array(y_test,  dtype=object)
 
# this is the location of the well known URL of the file
np.savez(f"{webroot}/{filename}", x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test)


  0%|          | 0/20 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
import requests
import os
import errno 

class Birdsongs(DatasetClient):
    def __init__(self):
        super().__init__()
            
    def load_data(self, ):
        url = "http://localhost:5001/birdsongs.npz"
        fname = "birdsongs.npz"
        cached_file= self.get_or_download_file(fname, url)
        print("loading")
 
        if(cached_file):
            data= np.load(f"{cached_file}", allow_pickle=True)
            x_train = data['x_train']
            y_train = data['y_train']
            x_test  = data['x_test']
            y_test  = data['y_test']
            return (x_train, y_train), (x_test, y_test)
        else:
            raise FileNotFoundError(errno.ENOENT, os.strerror(errno.ENOENT), filename)
        
bs =Birdsongs() 
try:
    bs.load_data()
except Exception as e:
    print(e)

In [ ]:

class DatasetClient():
 
    def __init__(self, app_name="navsea_ml", app_author="navsea"):
        # Get the cache directory path
        self.app_name = app_name
        self.app_author= app_author
        self.cache_dir = appdirs.user_cache_dir(appname=self.app_name , appauthor=self.app_author )
        
        # Check if the cache directory exists, create it if not
        if not os.path.exists(self.cache_dir):
            try:
                os.makedirs(self.cache_dir)
                print(f"Cache directory created: {self.cache_dir}")
            except OSError as e:
                print(f"Failed to create cache directory: {self.cache_dir}")
                raise  # Handle the exception as needed
        print(f"Cache directory path: {self.cache_dir}")

    def get_or_download_file(self, filename, url):
        file_path = os.path.join(self.cache_dir, filename)

        # Check if the file already exists in the cache
        if os.path.exists(file_path):
            print(f"Using cached file: {file_path}")
            return file_path

        # If not, download it from the URL
        print(f"Downloading {filename} from {url}...")
        response = requests.get(url)

        # Check if the download was successful
        if response.status_code == 200:
            with open(file_path, 'wb') as f:
                f.write(response.content)
            print(f"File downloaded and saved to: {file_path}")
            return file_path
        else:
            print(f"Failed to download {filename}. Status code: {response.status_code}")
            return None
        
    def load_data(self):
        # this is to be overloaded in the derived class to 
        # implement the processing specific to this data type
        pass


ds = DatasetClient()
ds.get_or_download_file("birddsongs.npz", "http://localhost:5001/birdsongs.npz")

In [ ]:
response = requests.post("http://127.0.0.1:5000/issue_api_key")
print(response.text)

In [ ]:
headers = {"X-API-Key":"bRx7LXfbH9RxRwb0h53JZ93Vi7OQA3OZbpw9esL2W8A"}
response = requests.get("http://127.0.0.1:5000/validate_api_key", headers=headers)
response.text

In [ ]:
response = requests.get("http://127.0.0.1:5000/mydataset.npz", headers=headers)